# NB 66 — multi-FRH vs subspace at the best λ, custom + WordNet concepts

A method-primary comparison of **multi-FRH** vs **subspace** steering across the
same joy/sorrow/safe/harm battery (4 singles + 4 repels + 4 blends), built from
**both** concept sources:

- **custom** — the curated English word lists (`core`).
- **wordnet** — the original FRH 8-language WordNet synsets (`wn8lang`).

at `λ = 0` and the **best λ per method**. From the fluency-dial evidence:

- **multi-FRH best λ = 0.4** — λ lifts *both* steering and fluency (NB 64; the
  FRH λ≈0.4 recipe).
- **subspace best λ = 0.4** — subspace has a pure trade-off (steering falls
  monotonically with λ), and λ=0.4 is the fluency knee: the ppl minimum in the
  useful range, beyond which both steering and fluency degrade (NB 61, pooled).

Both best λ land at 0.4, so this reuses the NB 65 battery
(`resources/65_source_battery.jsonl`, logged at λ∈{0,0.4}) — **no new
generation**, just the method-primary re-analysis.

**Reading the axes.** Directed effect is in each method's own expression units
(frame trace-correlation vs subspace ‖B·h‖), so it is *not* comparable across
methods — read it within a (method, source) to gauge steering strength. The
head-to-head **mfrh-vs-subspace** verdict rests on the neutral, shared-unit axes:
**lexical success** (presence of a core word) and the **perplexity ratio**.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
SRC_JSONL = "resources/65_source_battery.jsonl"  # NB65 battery (λ∈{0,0.4})
SUFFIX = ""
BEST_LAM = {"multifrh": 0.4, "subspace": 0.4}  # best λ per method (see header)
SOURCE_LABEL = {"core": "custom", "wn8lang": "wordnet"}
METHODS = ["multifrh", "subspace"]
SOURCES = ["custom", "wordnet"]
CASES = ["1-concept", "2-concept"]

In [ ]:
records = [json.loads(x) for x in Path(SRC_JSONL).read_text().strip().split("\n") if x]
df = pd.DataFrame([
    {"method": r["config"]["method"],
     "source": SOURCE_LABEL[r["config"]["source"]],
     "case": r["config"]["case"], "kind": r["config"]["kind"],
     "name": r["config"]["name"], "lam": r["config"]["lam"],
     "directed": r["directed_effect"], "success": r["success"],
     "ppl_ratio": r["ppl_ratio"]}
    for r in records
])
# keep λ=0 and each method's best λ
df = df[df.apply(lambda x: x["lam"] in (0.0, BEST_LAM[x["method"]]), axis=1)].copy()
df["lam_tag"] = df.apply(lambda x: "0" if x["lam"] == 0.0 else "best", axis=1)
print(f"{len(df)} generations | methods={sorted(df.method.unique())} "
      f"sources={sorted(df.source.unique())} λ={sorted(df.lam.unique())}")

In [ ]:
summary = (
    df.groupby(["source", "case", "method", "lam_tag"])
    .agg(directed_mean=("directed", "mean"),
         success_rate=("success", "mean"),
         ppl_ratio=("ppl_ratio", "mean"),
         n=("directed", "size"))
    .reset_index()
    .sort_values(["source", "case", "method", "lam_tag"])
)
summary.to_csv(f"resources/66_mfrh_vs_subspace_summary{SUFFIX}.csv", index=False)
print("=== mfrh vs subspace, per source / case / λ ===")
print(summary.round(4).to_string(index=False))

In [ ]:
# Head-to-head on the neutral axes: mfrh − subspace (positive = mfrh better
# success / mfrh higher ppl). At the best λ per method.
best = summary[summary.lam_tag == "best"]
h2h = best.pivot_table(index=["source", "case"], columns="method",
                       values=["success_rate", "ppl_ratio"])
h2h[("success_rate", "mfrh−sub")] = h2h[("success_rate", "multifrh")] - h2h[("success_rate", "subspace")]
h2h[("ppl_ratio", "mfrh−sub")] = h2h[("ppl_ratio", "multifrh")] - h2h[("ppl_ratio", "subspace")]
print("=== head-to-head at best λ (neutral axes) ===")
print(h2h.round(4).to_string())

In [ ]:
# Plot at the best λ: rows = source (custom, wordnet), cols = neutral axes
# (success, ppl) + within-method steering strength; bars grouped by method.
mcolors = {"multifrh": "tab:green", "subspace": "tab:blue"}
panels = [("success_rate", "core-word success", None),
          ("ppl_ratio", "perplexity ratio", 2.5),
          ("directed_mean", "steering (self-ref,\nper-method units)", None)]
x = np.arange(len(CASES)); width = 0.38
fig, axes = plt.subplots(len(SOURCES), len(panels), figsize=(4 * len(panels), 3.4 * len(SOURCES)))
for row, src in enumerate(SOURCES):
    for col, (metric, label, guard) in enumerate(panels):
        ax = axes[row, col]
        for j, m in enumerate(METHODS):
            vals = [best[(best.source == src) & (best.case == c) & (best.method == m)][metric].mean()
                    for c in CASES]
            ax.bar(x + (j - 0.5) * width, vals, width, label=m, color=mcolors[m])
        ax.set_xticks(x); ax.set_xticklabels(CASES)
        ax.set_title(f"{src} — {label}")
        ax.axhline(0, color="gray", lw=0.6)
        if guard:
            ax.axhline(guard, color="red", ls=":", lw=0.8)
        ax.legend(fontsize=8)
plt.suptitle("multi-FRH vs subspace at best λ (=0.4) — custom vs WordNet concepts")
plt.tight_layout()
plt.savefig(f"resources/66_mfrh_vs_subspace{SUFFIX}.png", dpi=150)
plt.show()

## Notes

- **Best λ = 0.4 for both methods** — multi-FRH because λ lifts both steering and
  fluency; subspace because 0.4 is its fluency knee (steering falls with λ, so
  no λ improves both — NB 61). λ=0 is the max-steering reference.
- **Cross-method verdict** uses the neutral axes only: `success_rate` (core-word
  presence) and `ppl_ratio`. Directed effect is per-method units — compare it
  *within* a (method, source), not across methods.
- Reuses `65_source_battery.jsonl`; no generation. Re-run NB 65 (custom concept
  battery) upstream to refresh the inputs.